In [130]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [131]:
import pandas as pd
df_train = pd.read_csv('/content/drive/MyDrive/train_data_90.csv')
df_test = pd.read_csv('/content/drive/MyDrive/test_data_90.csv')

In [132]:
def check_mixed_types(df, column_name):
    """Checks if a column in a DataFrame has mixed data types."""

    unique_types = df[column_name].apply(type).unique()
    # print(unique_types)
    return len(unique_types)

def drop_rows_with_float(df, column_name):
    """Drops rows where the specified column has float type data."""
    df_filtered = df[df[column_name].apply(type) != float ]
    return df_filtered



#FILTER TRAIN SET

In [133]:
features = df_train.columns
for feature in features:
    mixed_types = check_mixed_types(df_train, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")

df_filtered = drop_rows_with_float(df_train, 'maritalstatus')
df_filtered = drop_rows_with_float(df_filtered, 'race')
df_filtered = drop_rows_with_float(df_filtered, 'sex')

df_train_filtered = df_filtered
features = df_filtered.columns
for feature in features:
    mixed_types = check_mixed_types(df_train_filtered, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")
df_train_filtered.shape

Column 'maritalstatus' has '2' data types.
Column 'race' has '2' data types.
Column 'sex' has '2' data types.


(26366, 14)

#FILTER TEST SET

In [134]:
df_filtered = drop_rows_with_float(df_test, 'maritalstatus')
df_filtered = drop_rows_with_float(df_filtered, 'race')
df_filtered = drop_rows_with_float(df_filtered, 'sex')

df_test_filtered = df_filtered
features = df_filtered.columns
for feature in features:
    mixed_types = check_mixed_types(df_test_filtered, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")



In [135]:
df_train.isna().sum()

,0
age,0
workclass,0
education,0
educationno,0
maritalstatus,179
occupation,0
relationship,0
race,606
sex,303
capitalgain,0


In [136]:
df_train['race'].value_counts()

,count
race,
White,22820
Black,2454
Asian-Pac-Islander,799
Amer-Indian-Eskimo,260
Other,205


In [137]:
categorical_features = ['workclass', 'education', 'maritalstatus', 'occupation', 'relationship', 'race', 'sex', 'native']
continuous_features = ['age', 'educationno', 'capitalgain', 'capitalloss', 'hoursperweek']
ohe_features = ['maritalstatus','race','sex']
fe_features = ['workclass','education','occupation','relationship','native']

In [138]:
df_train['occupation'].value_counts()

,count
occupation,
Craft-repair,3648
Prof-specialty,3643
Exec-managerial,3593
Adm-clerical,3349
Sales,3241
Other-service,2850
Machine-op-inspct,1762
Transport-moving,1406
Handlers-cleaners,1220


# ONE HOT ENCODING

##OHE for Train set


In [139]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# --- One-Hot Encoding ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_encoded = pd.DataFrame(ohe.fit_transform(df_train_filtered[ohe_features]))
ohe_encoded.columns = ohe.get_feature_names_out(ohe_features)
# --- Frequency Encoding ---
fe_encoded = df_train_filtered[fe_features].apply(lambda x: x.map(x.value_counts(normalize=True)))
# --- Combine and Drop ---
df_train_filtered = df_train_filtered.drop(ohe_features + fe_features, axis=1)
df_train_filtered = pd.concat([df_train_filtered, ohe_encoded, fe_encoded], axis=1)

##OHE for Test set

In [140]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# --- One-Hot Encoding ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_encoded = pd.DataFrame(ohe.fit_transform(df_test_filtered[ohe_features]))
ohe_encoded.columns = ohe.get_feature_names_out(ohe_features)

# --- Frequency Encoding ---
fe_encoded = df_test_filtered[fe_features].apply(lambda x: x.map(x.value_counts(normalize=True)))

# --- Combine and Drop ---
df_test_filtered = df_test_filtered.drop(ohe_features + fe_features, axis=1)
df_test_filtered = pd.concat([df_test_filtered, ohe_encoded, fe_encoded], axis=1)

#Split into X and y

In [141]:
X_train=df_train_filtered.drop(['Possibility','educationno'],axis=1)
y_train=df_train_filtered['Possibility']
X_test=df_test_filtered.drop(['Possibility','educationno'],axis=1)
y_test=df_test_filtered['Possibility']

In [142]:
X_train.shape,y_train.shape

((27118, 23), (27118,))

In [143]:
df2=pd.concat([X_train,y_train],axis=1)

In [144]:
df2=df2.drop_duplicates()

In [145]:
df2_test=pd.concat([X_test,y_test],axis=1)

In [146]:
df2_test=df2_test.drop_duplicates()

In [147]:
df2.shape,df2_test.shape

((25275, 24), (2920, 24))

In [148]:
df2['profit']=df2['capitalgain']-df2['capitalloss']
df2_test['profit']=df2_test['capitalgain']-df2_test['capitalloss']
df2=df2.drop(['capitalgain','capitalloss'],axis=1)
df2_test=df2_test.drop(['capitalgain','capitalloss'],axis=1)

In [149]:
df2.isna().sum()

,0
age,39
hoursperweek,447
maritalstatus_Divorced,745
maritalstatus_Married-AF-spouse,745
maritalstatus_Married-civ-spouse,745
maritalstatus_Married-spouse-absent,745
maritalstatus_Never-married,745
maritalstatus_Separated,745
maritalstatus_Widowed,745
race_Amer-Indian-Eskimo,745


In [150]:
df2_test.isna().sum()

,0
age,17
hoursperweek,63
maritalstatus_Divorced,91
maritalstatus_Married-AF-spouse,91
maritalstatus_Married-civ-spouse,91
maritalstatus_Married-spouse-absent,91
maritalstatus_Never-married,91
maritalstatus_Separated,91
maritalstatus_Widowed,91
race_Amer-Indian-Eskimo,91


In [151]:
df2 = df2.dropna(subset=['Possibility'])
df2_test = df2_test.dropna(subset=['Possibility'])

In [152]:
X_train=df2.drop('Possibility',axis=1)
y_train=df2['Possibility']
X_test=df2_test.drop('Possibility',axis=1)
y_test=df2_test['Possibility']

In [153]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((25236, 22), (25236,), (2903, 22), (2903,))

##Displaying first five rows with Nan Values

In [154]:
null_rows = X_train[X_train.isnull().any(axis=1)]
if not null_rows.empty:
  print(null_rows.head(5))
else:
  print("No rows with null values found.")

      age  hoursperweek  maritalstatus_Divorced  \
12   35.0           NaN                     0.0   
19   29.0           NaN                     0.0   
53   41.0           NaN                     0.0   
101  46.0           NaN                     0.0   
160  50.0           NaN                     1.0   

     maritalstatus_Married-AF-spouse  maritalstatus_Married-civ-spouse  \
12                               0.0                               0.0   
19                               0.0                               0.0   
53                               0.0                               1.0   
101                              0.0                               1.0   
160                              0.0                               0.0   

     maritalstatus_Married-spouse-absent  maritalstatus_Never-married  \
12                                   0.0                          0.0   
19                                   0.0                          1.0   
53                           

##Filled Nan values with respective median values

In [155]:
for column in X_train.columns:
  if column == 'hoursperweek':
    X_train[column].fillna(0, inplace=True) # Fill NaNs in 'hoursperweek' with 0
  else:
    X_train[column].fillna(X_train[column].median(), inplace=True) # Fill NaNs in other columns with median

In [156]:
null_counts = y_train.isna().sum()
print(null_counts)

0


In [157]:
for column in X_test.columns:
  if column == 'hoursperweek':
    X_test[column].fillna(0, inplace=True) # Fill NaNs in 'hoursperweek' with 0
  else:
    X_test[column].fillna(X_test[column].median(), inplace=True)


In [158]:
null_counts_test = X_test.isna().sum()
print(null_counts_test)

age                                    0
hoursperweek                           0
maritalstatus_Divorced                 0
maritalstatus_Married-AF-spouse        0
maritalstatus_Married-civ-spouse       0
maritalstatus_Married-spouse-absent    0
maritalstatus_Never-married            0
maritalstatus_Separated                0
maritalstatus_Widowed                  0
race_Amer-Indian-Eskimo                0
race_Asian-Pac-Islander                0
race_Black                             0
race_Other                             0
race_White                             0
sex_Female                             0
sex_Male                               0
workclass                              0
education                              0
occupation                             0
relationship                           0
native                                 0
profit                                 0
dtype: int64


##Scaling the processed train and test data with MinMaxScaler

In [159]:
common_features = X_train.columns.intersection(X_test.columns)
X_train = X_train[common_features]
X_test = X_test[common_features]
X_test = X_test[X_train.columns]
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [160]:
X_test_scaled

array([[1.50684932e-01, 5.55555556e-01, 0.00000000e+00, ...,
        6.39135836e-01, 1.00330627e+00, 4.17421302e-02],
       [3.28767123e-01, 4.04040404e-01, 0.00000000e+00, ...,
        9.92953495e-01, 7.11569140e-04, 4.17421302e-02],
       [4.93150685e-01, 5.05050505e-01, 0.00000000e+00, ...,
        9.92953495e-01, 5.60044052e-03, 4.17421302e-02],
       ...,
       [9.58904110e-02, 4.04040404e-01, 0.00000000e+00, ...,
        6.39135836e-01, 1.00330627e+00, 4.17421302e-02],
       [5.47945205e-02, 4.04040404e-01, 0.00000000e+00, ...,
        6.39135836e-01, 4.09617240e-03, 4.17421302e-02],
       [9.58904110e-02, 4.04040404e-01, 0.00000000e+00, ...,
        6.39135836e-01, 1.00330627e+00, 4.17421302e-02]])

In [161]:
X_test_scaled.shape,y_test.shape,X_train_scaled.shape,y_train.shape

((2903, 22), (2903,), (25236, 22), (25236,))

## Finding Feature importances using Random Forest for further processing

In [162]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train_scaled, y_train)
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df)

                                Feature  Importance
0                                   age    0.217605
21                               profit    0.158467
19                         relationship    0.156677
18                           occupation    0.116697
1                          hoursperweek    0.106466
17                            education    0.091231
16                            workclass    0.044467
20                               native    0.019480
4      maritalstatus_Married-civ-spouse    0.013360
6           maritalstatus_Never-married    0.013317
15                             sex_Male    0.009439
14                           sex_Female    0.009262
2                maritalstatus_Divorced    0.009161
13                           race_White    0.008157
11                           race_Black    0.007141
7               maritalstatus_Separated    0.004391
8                 maritalstatus_Widowed    0.004303
10              race_Asian-Pac-Islander    0.003695
5   maritals

In [163]:
important_features=['age','profit','relationship','occupation','hoursperweek','education','workclass']
X_train_imp=X_train_scaled[:,[X_train.columns.get_loc(c) for c in important_features]] # Get the integer index of each column name and use that to slice the NumPy array.
X_test_imp=X_test_scaled[:,[X_test.columns.get_loc(c) for c in important_features]] # Get the integer index of each column name and use that to slice the NumPy array.

## Sklearn KNN

In [179]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Create a KNN classifier object
knn = KNeighborsClassifier(n_neighbors=20)

# Perform 5-fold cross-validation for accuracy
accuracy_scores = cross_val_score(knn, X_train_imp, y_train, cv=5, scoring='accuracy')
print("Cross-validation accuracy scores:", accuracy_scores)
print(f"Average accuracy: {accuracy_scores.mean()}")

# Perform 5-fold cross-validation for F1-score
knn.fit(X_train_imp,y_train)
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

Cross-validation accuracy scores: [0.82012678 0.82662968 0.82187438 0.80760848 0.82246879]
Average accuracy: 0.819741623666222
F1 score: 0.8116769173735736


## Sklearn Random Forest Classifier

In [115]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Initialize the Random Forest model with some good hyperparameters
rf_model = RandomForestClassifier(
    n_estimators=150,
    criterion='gini',
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42
)

# Perform 5-fold cross-validation
scores = cross_val_score(rf_model, X_train_imp, y_train, cv=10, scoring='accuracy')

# Print the cross-validation scores and the average accuracy
print("Cross-validation scores:", scores)
print(f"Average accuracy: {scores.mean()}")
f1_scores = cross_val_score(rf_model, X_train_imp, y_train, cv=5, scoring='f1')
print(f"Average F1 score: {f1_scores.mean()}")

Cross-validation scores: [0.85974643 0.86172742 0.86172742 0.86093502 0.85380349 0.85380349
 0.843044   0.83987317 0.85889814 0.85334919]
Average accuracy: 0.8546907751381427
Average F1 score: 0.6656703864717441


##XGB with Optimal Paramters

In [41]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# Initialize the XGBoost model with some good hyperparameters
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42
)

scores = cross_val_score(xgb_model, X_train_imp, y_train, cv=5, scoring='accuracy')

print("Cross-validation scores:", scores)
print(f"Average accuracy: {scores.mean()}")
f1_scores = cross_val_score(xgb_model, X_train_imp, y_train, cv=5, scoring='f1')
print(f"Average F1 score: {f1_scores.mean()}")

Cross-validation scores: [0.87341521 0.87061621 0.86804042 0.85357638 0.86724787]
Average accuracy: 0.8665792187353301
Average F1 score: 0.7095730025022245


##Randomized Search with RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid
param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

# Create a Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the model
random_search.fit(X_train_imp, y_train)

# Get the best parameters and the best score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

# Get the best model
best_rf_model = random_search.best_estimator_

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters found:  {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 10, 'bootstrap': False}
Best accuracy found:  0.8558242733525627


##Voting of RandomForest and KNN

In [42]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

rf_model = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=15)
knn_model = KNeighborsClassifier(n_neighbors=20)

ensemble_model = VotingClassifier(estimators=[
    ('rf', rf_model), ('knn', knn_model)
], voting='soft')

# Fit the ensemble model
ensemble_model.fit(X_train_imp, y_train)

# Predict and calculate accuracy
y_pred = ensemble_model.predict(X_test_imp)
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy (RF and KNN): {accuracy}")
f1_scores = cross_val_score(ensemble_model, X_train_imp, y_train, cv=5, scoring='f1')
print(f"Average F1 score: {f1_scores.mean()}")

Ensemble Model Accuracy (RF and KNN): 0.8463658284533242
Average F1 score: 0.6675787526021437


## Voting of RF, XGB and KNN

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# Initialize the individual models
rf_model = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=15)
knn_model = KNeighborsClassifier(n_neighbors=20)
xgb_model = XGBClassifier(
     n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1
)

ensemble_model = VotingClassifier(estimators=[
    ('rf', rf_model), ('knn', knn_model), ('xgb', xgb_model)
], voting='soft')

# Fit the ensemble model
ensemble_model.fit(X_train_imp, y_train)

# Predict and calculate accuracy
y_pred = ensemble_model.predict(X_test_imp)
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy (RF, KNN, XGBoost): {accuracy}")

Ensemble Model Accuracy (RF, KNN, XGBoost): 0.8418869880767237


## AdaBoost from Scratch

In [175]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

class AdaBoostClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator, n_estimators=200,learning_rate=0.01):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.estimators_ = []
        self.estimator_weights_ = np.zeros(self.n_estimators, dtype=np.float64)
        self.estimator_errors_ = np.ones(self.n_estimators, dtype=np.float64)

    def fit(self, X, y):
        n_samples = X.shape[0]
        sample_weights = np.full(n_samples, (1 / n_samples))

        for i in range(self.n_estimators):
            estimator = self.base_estimator
            estimator.fit(X, y, sample_weight=sample_weights)

            y_pred = estimator.predict(X)
            incorrect = (y_pred != y)
            error = np.dot(incorrect, sample_weights) / np.sum(sample_weights)
            decay=0.99
            alpha = self.learning_rate * (decay ** i) * 0.5 * np.log((1.0 - error) / error)
            sample_weights *= np.exp(alpha * incorrect * ((sample_weights > 0) | (alpha < 0)))

            self.estimators_.append(estimator)
            self.estimator_weights_[i] = alpha
            self.estimator_errors_[i] = error

        return self
    def get_params(self, deep=True):
      # suppose this estimator has parameters "alpha" and "recursive"
      return {"base_estimator": self.base_estimator, "n_estimators": self.n_estimators, "learning_rate": self.learning_rate}

    def set_params(self, **parameters):
      for parameter, value in parameters.items():
          setattr(self, parameter, value)
      return self
    def predict(self, X):
        n_samples = X.shape[0]
        y_pred = np.zeros((n_samples, 1))

        for i, estimator in enumerate(self.estimators_):
            y_pred_i = estimator.predict(X).reshape((n_samples, 1))
            y_pred += self.estimator_weights_[i] * y_pred_i

        y_pred = np.sign(y_pred).flatten()
        return y_pred


from sklearn.tree import DecisionTreeClassifier
X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test
base_estimator = DecisionTreeClassifier(max_depth=5)
adaboost_model = AdaBoostClassifier(base_estimator=base_estimator, n_estimators=150)
adaboost_model.fit(X_train, y_train)

y_pred = adaboost_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"AdaBoost Accuracy: {accuracy}")
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

AdaBoost Accuracy: 0.8239751980709611
F1 score: 0.8196532140721444


##Gradient Boosting From Scratch

In [174]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from sklearn.metrics import make_scorer, accuracy_score

class GradientBoostingClassifier:
    def __init__(self, n_estimators=200, learning_rate=0.05, max_depth=10):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        # Initialize with log(odds)
        self.initial_prediction = np.log(np.sum(y == 1) / np.sum(y == 0))
        predictions = np.full(len(X), self.initial_prediction)

        for _ in range(self.n_estimators):
            # Calculate negative gradient (residuals)
            residuals = y - 1 / (1 + np.exp(-predictions))

            # Fit a tree to the residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)

            # Update predictions
            predictions += self.learning_rate * tree.predict(X)

    def predict(self, X):
        predictions = np.full(len(X), self.initial_prediction)

        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        # Convert log(odds) to probabilities and then to class labels
        probabilities = 1 / (1 + np.exp(-predictions))
        return (probabilities > 0.5).astype(int)
    def get_params(self, deep=True):
        return {
            'n_estimators': self.n_estimators,
            'learning_rate': self.learning_rate,
            'max_depth': self.max_depth,
        }

    # Add set_params method
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

# Define a custom scorer for cross-validation
def accuracy_scorer(model, X, y):
    y_pred = model.predict(X)
    return accuracy_score(y, y_pred)

X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test

# Initialize and evaluate the model with cross-validation
gb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=10)
gb_model.fit(X_train, y_train)
y_pred = gb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"GradientBoost: {accuracy}")
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

GradientBoost: 0.8573889080261798
F1 score: 0.8511739432779659


In [ ]:
import pandas as pd
!pip install pandas

# Concatenate X_train_imp and y_train
Train_final = pd.concat([pd.DataFrame(X_train_imp).reset_index(drop=True), pd.DataFrame(y_train).reset_index(drop=True)], axis=1)

# Save to CSV
Train_final.to_csv('Train_final.csv', index=False)
Test_final = pd.concat([pd.DataFrame(X_test_imp).reset_index(drop=True), pd.DataFrame(y_test).reset_index(drop=True)], axis=1)

# Save to CSV
Test_final.to_csv('Test_final.csv', index=False)

In [ ]:
Train_final.isna().sum()

,0
0,0
1,0
2,0
3,0
4,0
5,0
6,0
Possibility,0


##Sklearn Decision Tree

In [118]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
dc=DecisionTreeClassifier(criterion='gini',min_samples_split=5,min_samples_leaf=3,max_features='sqrt',max_depth=10)
dc.fit(X_train_imp,y_train)


DecisionTreeClassifier(max_depth=10, max_features='sqrt', min_samples_leaf=3,
                       min_samples_split=5)

In [173]:
y_pred=dc.predict(X_test_imp)
accuracy=accuracy_score(y_test,y_pred)
print(f"Accuracy:{accuracy}")
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

Accuracy:0.8318980365139511
F1 score: 0.8199491659229119


##Sklearn SVC

In [120]:
from sklearn.svm import SVC
sv=SVC(kernel='rbf',C=3,gamma='scale',class_weight='balanced',degree=5)
sv.fit(X_train_imp,y_train)

SVC(C=3, class_weight='balanced', degree=5)

In [171]:
y_pred=sv.predict(X_test_imp)
accuracy=accuracy_score(y_test,y_pred)
print(f"Accuracy:{accuracy}")
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

Accuracy:0.7636927316569067
F1 score: 0.7778895627520043


##Scratch Naive Bayes

In [170]:
import numpy as np
from collections import Counter
from imblearn.over_sampling import ADASYN
from sklearn.metrics import accuracy_score, f1_score

class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing  # Variance smoothing parameter

    def fit(self, X, y):
        """
        Fit the Gaussian Naive Bayes model to the training data.
        """
        self.classes = np.unique(y)
        self.means = {}
        self.variances = {}
        self.priors = {}

        for c in self.classes:
            X_c = X[y == c]
            self.means[c] = np.mean(X_c, axis=0)
            self.variances[c] = np.var(X_c, axis=0) + self.var_smoothing  # Add var_smoothing to variance
            self.priors[c] = X_c.shape[0] / float(X.shape[0])

    def _predict_log_proba(self, X):
        """
        Calculate the log probability of each class for each sample.
        """
        log_probs = []
        for c in self.classes:
            mean = self.means[c]
            variance = self.variances[c]
            prior = np.log(self.priors[c])
            # Compute the log probability of the data given the class
            log_prob = -0.5 * np.sum(np.log(2. * np.pi * variance))
            log_prob -= 0.5 * np.sum(((X - mean) ** 2) / variance, axis=1)
            log_probs.append(log_prob + prior)
        return np.array(log_probs).T

    def predict(self, X):
        """
        Predict class labels for the input data.
        """
        log_probs = self._predict_log_proba(X)
        return self.classes[np.argmax(log_probs, axis=1)]

    def predict_proba(self, X):
        """
        Predict class probabilities for the input data.
        """
        log_probs = self._predict_log_proba(X)
        # Convert log probabilities to probabilities
        exp_log_probs = np.exp(log_probs - np.max(log_probs, axis=1, keepdims=True))
        return exp_log_probs / np.sum(exp_log_probs, axis=1, keepdims=True)

    def get_params(self, deep=True):
        return {"var_smoothing": self.var_smoothing}

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self


# Assuming X_train, X_test, y_train, y_test are defined
X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test

# Apply ADASYN to balance the training data
adasyn = ADASYN(n_neighbors=5,random_state=42)
X_train_balanced, y_train_balanced = adasyn.fit_resample(X_train, y_train)

# Initialize and fit the Gaussian Naive Bayes model
model = GaussianNaiveBayes(var_smoothing=1e-9)
model.fit(X_train_balanced, y_train_balanced)

# Predict on the test set
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred,average='weighted')

# Print evaluation metrics
print(f"Gaussian Naive Bayes Accuracy: {accuracy}")
print(f"Gaussian Naive Bayes F1 score: {f1}")


Gaussian Naive Bayes Accuracy: 0.7977953840854288
Gaussian Naive Bayes F1 score: 0.767530430344599


##Sklearn Gaussian NB

In [169]:
from sklearn.naive_bayes import GaussianNB
model=GaussianNB()
X_train,y_train=X_train_imp,y_train
model.fit(X_train_imp,y_train)
y_pred=model.predict(X_test_imp)
accuracy=accuracy_score(y_test,y_pred)
print(f"Accuracy:{accuracy}")
f1 = f1_score(y_test, y_pred,average='weighted')
print(f"F1 score: {f1}")

Accuracy:0.7881501894591801
F1 score: 0.7397241328215985


##Voting from Scratch for Scratch Gradient Boosting ,Scratch Naive Bayes ,sklearn KNN,SVC and Decision Tree

In [168]:
class VotingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, classifiers):
        self.classifiers = classifiers

    def fit(self, X, y):
        for clf in self.classifiers:
            clf.fit(X, y)
        return self

    def predict(self, X):
        predictions = np.array([clf.predict(X) for clf in self.classifiers])
        predictions = predictions.astype(int)
        majority_vote = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions)
        return majority_vote

    def predict_proba(self, X):
        probs = np.array([clf.predict_proba(X) for clf in self.classifiers])
        avg_probs = np.mean(probs, axis=0)
        return avg_probs

# Initialize individual classifiers
nb_clf = GaussianNaiveBayes(var_smoothing=1e-9)
svc_clf = SVC(kernel='rbf',class_weight='balanced',probability=True, random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=10)
dt_clf = DecisionTreeClassifier(
    criterion='gini',
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    max_depth=10,
    random_state=42
    )
gb_clf = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=10)

# Initialize the VotingClassifier
voting_clf = VotingClassifier(classifiers=[nb_clf, svc_clf, knn_clf, dt_clf, gb_clf])

X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test

# Fit the VotingClassifier
voting_clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = voting_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

# Print evaluation metrics
print(f"Voting Classifier Accuracy: {accuracy}")
print(f"Voting Classifier F1 score: {f1}")

Voting Classifier Accuracy: 0.853255253186359
Voting Classifier F1 score: 0.8460874566646628
